# Exploratory Data Analysis: BTC, Gold, and Silver

This notebook provides a visual and statistical deep dive into our cleaned historical data. We analyze price trends, trading volumes, and basic statistics for our three main assets.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import os

# Robust Path Handling: Check if we are running from 'notebooks' or root
if os.path.exists('../data/processed/'):
    PROCESSED_DIR = '../data/processed/'
elif os.path.exists('data/processed/'):
    PROCESSED_DIR = 'data/processed/'
else:
    raise FileNotFoundError("Could not find 'data/processed' directory. Please ensure the pipeline has been run.")

assets = ['bitcoin_data.csv', 'gold_data.csv', 'silver_data.csv']

# Load data
dfs = {f.split('_')[0].capitalize(): pd.read_csv(os.path.join(PROCESSED_DIR, f), parse_dates=['timestamp']) for f in assets}

print(f"Datasets loaded successfully from {PROCESSED_DIR}.")

## 1. Basic Statistical Summary

We calculate the distribution and scale of our data across all assets.

In [ ]:
for name, df in dfs.items():
    print(f"\n--- Stats for {name} ---")
    # Select only numeric columns for stats
    stats = df.select_dtypes(include=['float64', 'int64']).describe()
    print(stats.apply(lambda s: s.apply('{0:.2f}'.format)))
    print("-"*30)

## 2. Price History Analysis

Visualizing the price trends for Bitcoin, Gold, and Silver over time.

In [ ]:
fig = make_subplots(rows=3, cols=1, 
                    subplot_titles=("Bitcoin Price History", "Gold Price History", "Silver Price History"),
                    vertical_spacing=0.1)

row = 1
colors = ['#F7931A', '#D4AF37', '#C0C0C0'] # BTC Orange, Gold, Silver

for (name, df), color in zip(dfs.items(), colors):
    fig.add_trace(
        go.Scatter(x=df['timestamp'], y=df['price'], name=name, line=dict(color=color)),
        row=row, col=1
    )
    row += 1

fig.update_layout(height=900, title_text="Asset Price Comparison", showlegend=False, template='plotly_dark')
fig.show()

## 3. Trading Volume Analysis

Volume often precedes price action. Here we look for anomalies or spikes in trading activity.

In [ ]:
fig_vol = make_subplots(rows=3, cols=1, 
                        subplot_titles=("Bitcoin Volume", "Gold Volume", "Silver Volume"),
                        vertical_spacing=0.1)

row = 1
for (name, df), color in zip(dfs.items(), colors):
    fig_vol.add_trace(
        go.Bar(x=df['timestamp'], y=df['volume'], name=f"{name} Vol", marker_color=color, opacity=0.7),
        row=row, col=1
    )
    row += 1

fig_vol.update_layout(height=900, title_text="Trading Volume Analysis", showlegend=False, template='plotly_dark')
fig_vol.show()

## 4. Technical Indicators (SMA & EMA)

We transition from raw prices to trend analysis. Below, we visualize **Bitcoin** with its 30-day and 60-day Simple Moving Averages to identify medium and long-term trends.

In [ ]:
btc_df = dfs['Bitcoin']

fig_ma = go.Figure()

# Raw Price
fig_ma.add_trace(go.Scatter(x=btc_df['timestamp'], y=btc_df['price'], name='BTC Price', 
                         line=dict(color='rgba(255,255,255,0.3)', width=1)))

# SMA 30
fig_ma.add_trace(go.Scatter(x=btc_df['timestamp'], y=btc_df['SMA_30'], name='SMA 30 (Medium Term)', 
                         line=dict(color='#00f2ff', width=2)))

# SMA 60
fig_ma.add_trace(go.Scatter(x=btc_df['timestamp'], y=btc_df['SMA_60'], name='SMA 60 (Long Term)', 
                         line=dict(color='#ff007f', width=2)))

fig_ma.update_layout(title='Bitcoin Trend Analysis: Price vs Moving Averages', 
                  xaxis_title='Date', yaxis_title='Price (USD)',
                  template='plotly_dark', height=600)
fig_ma.show()

### EMA vs SMA Comparison

Exponential Moving Averages react faster to recent price changes compared to Simple Moving Averages.

In [ ]:
fig_ema = go.Figure()

fig_ema.add_trace(go.Scatter(x=btc_df['timestamp'], y=btc_df['SMA_14'], name='SMA 14', 
                          line=dict(dash='dash', color='gray')))
fig_ema.add_trace(go.Scatter(x=btc_df['timestamp'], y=btc_df['EMA_14'], name='EMA 14', 
                          line=dict(color='#00ff00')))

fig_ema.update_layout(title='Indicator Comparison: SMA vs EMA (14-Day)',
                   template='plotly_dark', height=400)
fig_ema.show()